# GWB Template Class — Demo

This notebook walks through the core features of the `Template` base class:

1. Browsing the registry and instantiating templates
2. Evaluating spectra with positional args or a parameter dict/array
3. Parameter metadata (names, labels, priors)
4. Derivatives (gradient, Hessian, frequency derivative)
5. JIT-compiling the hot path with `jax.jit`
6. Writing your own template subclass
7. Plotting a few built-in templates

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import gwb_templates
from gwb_templates import AnalyticTemplate, Template, get_template_from_registry

## 1. The registry

Importing `gwb_templates` eagerly loads every template module, which registers each concrete subclass.  
You can see all registered templates with `Template.registered_templates()`.

In [ ]:
registry = Template.registered_templates()
print(f"{len(registry)} templates registered:\n")
for name in sorted(registry):
    print(" ", name)

## 2. Instantiating a template

Two equivalent ways to get an instance:

In [ ]:
from gwb_templates.generic_templates.power_law import PowerLaw

# Direct instantiation
pl = PowerLaw()

# Or by name from the registry
pl2 = get_template_from_registry("PowerLaw")

print(pl)
print()
print("parameter names :", pl.parameter_names)
print("n_params         :", pl.n_params)
print("model_id         :", pl.model_id)
print("model_label      :", pl.model_label)
print("jittable         :", pl.jittable)
print("diff backend     :", pl.differentiation_backend)

### Parameter labels and priors

Each template ships with default LaTeX-ready labels and prior bounds.

In [ ]:
print("Labels:", dict(pl.parameter_labels))
print("Priors:", dict(pl.prior_by_param))

You can override labels and priors at construction time:

In [ ]:
pl_custom = PowerLaw(
    model_name="my_pl",
    parameter_labels={"tilt": r"$n_{\rm tilt}$"},
    prior_by_param={"log_amplitude": {"min": -15.0, "max": -8.0}, "tilt": {"min": -5.0, "max": 5.0}},
)
print(pl_custom)
print("Labels:", dict(pl_custom.parameter_labels))
print("Priors:", dict(pl_custom.prior_by_param))

## 3. Evaluating the spectrum

There are two calling conventions. Use whichever fits your pipeline.

### 3a. Direct call — spread positional args

In [ ]:
freqs = jnp.geomspace(1e-4, 1e-1, 200)  # Hz, LISA band

log_amp = -11.0
tilt    =   2.0

spectrum = pl.omega_gw_h2(freqs, log_amp, tilt)
print("spectrum shape:", spectrum.shape)
print("first value   :", spectrum[0])

### 3b. Vector / dict entry point — sampler-friendly

`omega_gw_h2_from_parameters` accepts either a dict or a plain 1-D array in `parameter_names` order.  
This is the canonical entry point to wrap in `jax.jit`.

In [ ]:
# Dict form
params_dict = {"log_amplitude": log_amp, "tilt": tilt}
s1 = pl.omega_gw_h2_from_parameters(freqs, params_dict)

# Array form  (must match parameter_names order)
params_vec = jnp.array([log_amp, tilt])
s2 = pl.omega_gw_h2_from_parameters(freqs, params_vec)

print("Dict vs array agree:", jnp.allclose(s1, s2))

## 4. Derivatives

Analytic templates use JAX autodiff by default.  
All derivative methods accept the same `(frequency, parameters)` signature.

In [ ]:
f0 = jnp.array(3e-3)  # single frequency

# Gradient w.r.t. parameters  →  shape (n_params,)
grad = pl.grad_theta_omega_gw_h2(f0, params_vec)
print("∂Ω/∂θ  :", grad)

# Hessian w.r.t. parameters  →  shape (n_params, n_params)
hess = pl.hess_theta_omega_gw_h2(f0, params_vec)
print("∂²Ω/∂θ²:", hess)

# Frequency derivative  →  shape () for scalar f
domega_df = pl.d_df_omega_gw_h2(f0, params_vec)
print("dΩ/df  :", domega_df)

In [ ]:
# All derivatives also accept frequency arrays
grad_array = pl.grad_theta_omega_gw_h2(freqs, params_vec)  # shape (200, 2)
print("∂Ω/∂θ over frequency band, shape:", grad_array.shape)

## 5. JIT compilation

Wrap `omega_gw_h2_from_parameters` in `jax.jit` once — subsequent calls hit the compiled version.  
(Only meaningful for `AnalyticTemplate`; `NumericalTemplate` sets `jittable=False`.)

In [ ]:
if pl.jittable:
    jit_spectrum = jax.jit(pl.omega_gw_h2_from_parameters)

    # Warm-up compile
    _ = jit_spectrum(freqs, params_vec).block_until_ready()

    import time
    t0 = time.perf_counter()
    for _ in range(500):
        jit_spectrum(freqs, params_vec).block_until_ready()
    elapsed_jit = (time.perf_counter() - t0) / 500

    t0 = time.perf_counter()
    for _ in range(500):
        pl.omega_gw_h2_from_parameters(freqs, params_vec).block_until_ready()
    elapsed_raw = (time.perf_counter() - t0) / 500

    print(f"JIT : {elapsed_jit*1e6:.1f} µs/call")
    print(f"Raw : {elapsed_raw*1e6:.1f} µs/call")

## 6. Writing your own template

Subclass `AnalyticTemplate`, implement `omega_gw_h2` with explicit positional parameters, and you're done.  
The class is auto-registered the moment it's defined — no manual registry call needed.

In [ ]:
class FlatSpectrum(AnalyticTemplate):
    """Simplest possible template: a flat (constant) spectrum."""

    def omega_gw_h2(self, frequency, log_amplitude):
        return jnp.full_like(jnp.asarray(frequency), 10.0**log_amplitude)


flat = FlatSpectrum()
print(flat)
print("parameter_names:", flat.parameter_names)
print("spectrum at 1 mHz:", flat.omega_gw_h2(1e-3, -12.0))

# It's in the registry automatically
print("\nIn registry:", "FlatSpectrum" in Template.registered_templates())

Keyword-only arguments with defaults are treated as *configuration*, not free parameters:

In [ ]:
class TiltedFlat(AnalyticTemplate):
    """Flat spectrum with an optional mild tilt fixed at construction."""

    def omega_gw_h2(self, frequency, log_amplitude, *, fixed_tilt=0.0, pivot=3e-3):
        return 10.0**log_amplitude * (jnp.asarray(frequency) / pivot) ** fixed_tilt


tf = TiltedFlat()
print("free params:", tf.parameter_names)  # only log_amplitude

## 7. Plotting built-in templates

A quick comparison of three generic templates across the LISA band.

In [ ]:
from gwb_templates.generic_templates.broken_power_law import BrokenPowerLaw
from gwb_templates.generic_templates.lognormal_bump import LognormalBump

freqs = jnp.geomspace(1e-4, 1e-1, 300)

# ── Power law ──────────────────────────────────────────────────────────────────
pl_spec = PowerLaw().omega_gw_h2(freqs, log_amplitude=-11.0, tilt=2.0)

# ── Broken power law ───────────────────────────────────────────────────────────
# Low-f tilt=2, high-f tilt=-1, break at 3 mHz, moderate transition (delta=1)
bpl_spec = BrokenPowerLaw().omega_gw_h2(
    freqs,
    log_amplitude=-11.0,
    log_pivot=float(jnp.log10(3e-3)),
    tilt_1=2.0,
    tilt_2=-1.0,
    log_transition=0.0,
)

# ── Log-normal bump ────────────────────────────────────────────────────────────
# Peak at 3 mHz, width sigma = 10^(-0.5) ≈ 0.32 decades
bump_spec = LognormalBump().omega_gw_h2(
    freqs,
    log_amplitude=-11.0,
    log_pivot=float(jnp.log10(3e-3)),
    log_width=-0.5,
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.loglog(freqs, pl_spec,   label=r"Power law ($n_T = 2$)")
ax.loglog(freqs, bpl_spec,  label=r"Broken power law ($n_1=2,\,n_2=-1$)", ls="--")
ax.loglog(freqs, bump_spec, label=r"Log-normal bump ($\sigma \approx 0.32$ dec)", ls=":")

ax.set_xlabel("Frequency [Hz]")
ax.set_ylabel(r"$\Omega_{\rm GW}\,h^2$")
ax.set_title("Built-in generic templates (same amplitude, same pivot)")
ax.legend()
ax.grid(True, which="both", ls=":")
plt.tight_layout()
plt.show()

## 8. Batching over parameter sets with `jax.vmap`

To evaluate many parameter sets at once, `vmap` over `omega_gw_h2_from_parameters`.

In [ ]:
rng = np.random.default_rng(0)
n_samples = 50
log_amps = rng.uniform(-13, -9, n_samples)
tilts    = rng.uniform(-1,   3, n_samples)
param_batch = jnp.column_stack([log_amps, tilts])  # shape (50, 2)

batched_eval = jax.vmap(lambda theta: pl.omega_gw_h2_from_parameters(freqs, theta))
spectra_batch = batched_eval(param_batch)  # shape (50, 300)
print("Batch output shape:", spectra_batch.shape)

fig, ax = plt.subplots(figsize=(8, 4))
for s in spectra_batch:
    ax.loglog(freqs, s, alpha=0.15, color="steelblue", lw=0.8)
ax.set_xlabel("Frequency [Hz]")
ax.set_ylabel(r"$\Omega_{\rm GW}\,h^2$")
ax.set_title(f"Power-law spectra for {n_samples} random parameter draws")
ax.grid(True, which="both", ls=":")
plt.tight_layout()
plt.show()